In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# NBVAL_IGNORE_OUTPUT
from openai.types.chat.chat_completion import Choice, ChoiceLogprobs
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_token_logprob import ChatCompletionTokenLogprob
from transformers import AutoTokenizer

import art
from art.preprocessing.tokenize import tokenize_trajectory_groups

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

In [3]:
# NBVAL_IGNORE_OUTPUT
prompt_token_ids = [
    151644,
    8948,
    198,
    2610,
    525,
    1207,
    16948,
    11,
    3465,
    553,
    54364,
    14817,
    13,
    1446,
    525,
    264,
    10950,
    17847,
    13,
    151645,
    198,
    151644,
    872,
    198,
    3838,
    374,
    279,
    6722,
    315,
    9625,
    30,
    151645,
    198,
    151644,
    77091,
    198,
]
tokenized_results = list(
    tokenize_trajectory_groups(
        tokenizer,
        [
            art.TrajectoryGroup(
                [
                    art.Trajectory(
                        messages_and_choices=[
                            {
                                "role": "user",
                                "content": "What is the capital of France?",
                            },
                            Choice.model_validate(
                                {
                                    "finish_reason": "stop",
                                    "index": 0,
                                    "logprobs": None,
                                    "message": ChatCompletionMessage(
                                        content="London",
                                        role="assistant",
                                    ),
                                    "prompt_token_ids": prompt_token_ids,
                                    "token_ids": [39572],
                                }
                            ),
                        ],
                        reward=0.0,
                    ),
                    art.Trajectory(
                        messages_and_choices=[
                            {
                                "role": "user",
                                "content": "What is the capital of France?",
                            },
                            Choice.model_validate(
                                {
                                    "finish_reason": "stop",
                                    "index": 0,
                                    "logprobs": ChoiceLogprobs(
                                        content=[
                                            ChatCompletionTokenLogprob(
                                                token="token:59604",
                                                bytes=[80, 97, 114, 105, 115],
                                                logprob=-0.01,
                                                top_logprobs=[],
                                            )
                                        ]
                                    ),
                                    "message": ChatCompletionMessage(
                                        content="Paris",
                                        role="assistant",
                                    ),
                                    "prompt_token_ids": prompt_token_ids,
                                    "token_ids": [59604],
                                }
                            ),
                        ],
                        reward=1.0,
                    ),
                ]
            )
        ],
        allow_training_without_logprobs=True,
        scale_rewards=True,
        shuffle_group_trajectories=False,
    )
)
actual = []
for result in tokenized_results:
    result.advantage = round(result.advantage, 2)
    result.weight = round(result.weight, 2)
    # set prompt_id to 0 to eliminate stochasticity
    result.prompt_id = 0
    actual.append(
        {
            "advantage": result.advantage,
            "token_ids": result.token_ids,
            "assistant_mask": result.assistant_mask,
            "nan_logprobs": sum(logprob != logprob for logprob in result.logprobs),
            "last_logprob": "nan"
            if result.logprobs[-1] != result.logprobs[-1]
            else result.logprobs[-1],
            "weight": result.weight,
            "prompt_id": result.prompt_id,
            "prompt_length": result.prompt_length,
        }
    )

assert actual == [
    {
        "advantage": -1.0,
        "token_ids": [
            151644,
            8948,
            198,
            2610,
            525,
            1207,
            16948,
            11,
            3465,
            553,
            54364,
            14817,
            13,
            1446,
            525,
            264,
            10950,
            17847,
            13,
            151645,
            198,
            151644,
            872,
            198,
            3838,
            374,
            279,
            6722,
            315,
            9625,
            30,
            151645,
            198,
            151644,
            77091,
            198,
            39572,
        ],
        "assistant_mask": [
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            1,
        ],
        "nan_logprobs": 37,
        "last_logprob": "nan",
        "weight": 1.0,
        "prompt_id": 0,
        "prompt_length": 35,
    },
    {
        "advantage": 1.0,
        "token_ids": [
            151644,
            8948,
            198,
            2610,
            525,
            1207,
            16948,
            11,
            3465,
            553,
            54364,
            14817,
            13,
            1446,
            525,
            264,
            10950,
            17847,
            13,
            151645,
            198,
            151644,
            872,
            198,
            3838,
            374,
            279,
            6722,
            315,
            9625,
            30,
            151645,
            198,
            151644,
            77091,
            198,
            59604,
        ],
        "assistant_mask": [
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            1,
        ],
        "nan_logprobs": 36,
        "last_logprob": -0.01,
        "weight": 1.0,
        "prompt_id": 0,
        "prompt_length": 35,
    },
]

TokenizedResult(advantage=-1.0, chat='<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\nLondon', tokens=['<|im_start|>', 'system', '\n', 'You', ' are', ' Q', 'wen', ',', ' created', ' by', ' Alibaba', ' Cloud', '.', ' You', ' are', ' a', ' helpful', ' assistant', '.', '<|im_end|>', '\n', '<|im_start|>', 'user', '\n', 'What', ' is', ' the', ' capital', ' of', ' France', '?', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n', 'London'], token_ids=[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 374, 279, 6722, 315, 9625, 30, 151645, 198, 151644, 77091, 198, 39572], input_pos=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36], assistant_mask=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0

TokenizedResult(advantage=1.0, chat='<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\nParis', tokens=['<|im_start|>', 'system', '\n', 'You', ' are', ' Q', 'wen', ',', ' created', ' by', ' Alibaba', ' Cloud', '.', ' You', ' are', ' a', ' helpful', ' assistant', '.', '<|im_end|>', '\n', '<|im_start|>', 'user', '\n', 'What', ' is', ' the', ' capital', ' of', ' France', '?', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n', 'Paris'], token_ids=[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 374, 279, 6722, 315, 9625, 30, 151645, 198, 151644, 77091, 198, 59604], input_pos=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36], assistant_mask=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0